*Nama: Leonard Dani Agustian*

*NIM: 2411513028*

*Jurusan: Teknik Komputer Kelas B*

In [1]:
# Persiapan library yang digunakan di seluruh notebook
import threading
import time
import subprocess
import sys
import tempfile
from pathlib import Path
from textwrap import dedent

print("Library siap digunakan.")

Library siap digunakan.


1. **RLock pada Class:** Buat sebuah class `RekeningBank` dengan method `setor(jumlah)` dan `tarik(jumlah)`, serta method `transfer(rekening_tujuan, jumlah)` yang memanggil `tarik()` pada dirinya sendiri dan `setor()` pada rekening tujuan. Gunakan `RLock` pada setiap instance. Jelaskan mengapa `RLock` diperlukan di sini dibanding `Lock` biasa.


In [2]:
# Definisi class RekeningBank menggunakan RLock pada setiap instance
class RekeningBank:
    def __init__(self, nama, saldo_awal=0):
        if saldo_awal < 0:
            raise ValueError("Saldo awal tidak boleh negatif.")

        self.nama = nama
        self.saldo = saldo_awal
        self._lock = threading.RLock()

    def setor(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah setoran harus lebih dari 0.")

        with self._lock:
            self.saldo += jumlah
            print(f"[{self.nama}] Setor Rp{jumlah:,} | Saldo: Rp{self.saldo:,}")
            return True

    def tarik(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah penarikan harus lebih dari 0.")

        with self._lock:
            if jumlah > self.saldo:
                print(f"[{self.nama}] Penarikan Rp{jumlah:,} gagal (saldo tidak cukup).")
                return False

            self.saldo -= jumlah
            print(f"[{self.nama}] Tarik Rp{jumlah:,} | Saldo: Rp{self.saldo:,}")
            return True

    def transfer(self, rekening_tujuan, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah transfer harus lebih dari 0.")
        if rekening_tujuan is self:
            print("Transfer ke rekening yang sama tidak dilakukan.")
            return False

        # Urutan lock dibuat konsisten untuk mencegah deadlock antar rekening.
        rekening_terkunci = sorted((self, rekening_tujuan), key=id)

        with rekening_terkunci[0]._lock:
            with rekening_terkunci[1]._lock:
                # tarik() mengambil kembali lock milik objek ini.
                if not self.tarik(jumlah):
                    print(f"Transfer dari {self.nama} ke {rekening_tujuan.nama} gagal.")
                    return False

                # setor() mengambil lock milik rekening tujuan.
                rekening_tujuan.setor(jumlah)
                print(f"Transfer Rp{jumlah:,} dari {self.nama} ke {rekening_tujuan.nama} berhasil.")
                return True


# Pengujian setor(), tarik(), dan transfer() pada dua rekening
rekening_a = RekeningBank("Rekening A", 100000)
rekening_b = RekeningBank("Rekening B", 50000)

rekening_a.setor(10000)
rekening_b.tarik(5000)

# Dua transfer dengan arah berlawanan dijalankan secara bersamaan.
t1 = threading.Thread(target=rekening_a.transfer, args=(rekening_b, 20000))
t2 = threading.Thread(target=rekening_b.transfer, args=(rekening_a, 10000))
t1.start()
t2.start()
t1.join()
t2.join()

print(f"\nSaldo akhir {rekening_a.nama}: Rp{rekening_a.saldo:,}")
print(f"Saldo akhir {rekening_b.nama}: Rp{rekening_b.saldo:,}")

[Rekening A] Setor Rp10,000 | Saldo: Rp110,000
[Rekening B] Tarik Rp5,000 | Saldo: Rp45,000
[Rekening A] Tarik Rp20,000 | Saldo: Rp90,000
[Rekening B] Setor Rp20,000 | Saldo: Rp65,000
Transfer Rp20,000 dari Rekening A ke Rekening B berhasil.
[Rekening B] Tarik Rp10,000 | Saldo: Rp55,000
[Rekening A] Setor Rp10,000 | Saldo: Rp100,000
Transfer Rp10,000 dari Rekening B ke Rekening A berhasil.

Saldo akhir Rekening A: Rp100,000
Saldo akhir Rekening B: Rp55,000


Penjelasan:

- `RLock` mengizinkan thread yang sama mengambil lock yang sama lebih dari satu kali. Saat `transfer()` sudah mengunci rekening sumber, method `tarik()` dipanggil dan mengambil lock rekening sumber tersebut sekali lagi. Karena itu, `RLock` diperlukan agar pemanggilan bertingkat ini tidak membuat thread mengunci dirinya sendiri.
- Jika menggunakan `Lock` biasa, pemanggilan `tarik()` dari dalam `transfer()` akan menunggu lock yang masih dipegang oleh thread itu sendiri (*self-deadlock*).
- Penguncian dua rekening dilakukan berdasarkan urutan `id()` agar transfer dengan arah berlawanan tetap mengambil lock dalam urutan yang konsisten.

2. **Deadlock dengan 3 Lock:** Buat skenario deadlock yang melibatkan tiga lock dan tiga thread dengan pola circular wait. Buktikan deadlock menggunakan `subprocess` dan `timeout`, kemudian perbaiki dengan urutan lock yang konsisten.


In [3]:
# Demonstrasi deadlock tiga thread dan tiga lock melalui subprocess + timeout
skrip_deadlock = dedent("""\
    import threading

    lock_a = threading.Lock()
    lock_b = threading.Lock()
    lock_c = threading.Lock()
    barrier = threading.Barrier(3)
    output_lock = threading.Lock()

    def tampilkan(pesan):
        with output_lock:
            print(pesan, flush=True)

    def thread1():
        with lock_a:
            tampilkan("Thread1 memegang Lock-A")
            barrier.wait()
            tampilkan("Thread1 menunggu Lock-B")
            with lock_b:
                tampilkan("Thread1 berhasil mengambil Lock-B")

    def thread2():
        with lock_b:
            tampilkan("Thread2 memegang Lock-B")
            barrier.wait()
            tampilkan("Thread2 menunggu Lock-C")
            with lock_c:
                tampilkan("Thread2 berhasil mengambil Lock-C")

    def thread3():
        with lock_c:
            tampilkan("Thread3 memegang Lock-C")
            barrier.wait()
            tampilkan("Thread3 menunggu Lock-A")
            with lock_a:
                tampilkan("Thread3 berhasil mengambil Lock-A")

    threads = [
        threading.Thread(target=thread1),
        threading.Thread(target=thread2),
        threading.Thread(target=thread3)
    ]

    for thread in threads:
        thread.start()

    for thread in threads:
        thread.join()
""")

with tempfile.TemporaryDirectory() as direktori:
    file_deadlock = Path(direktori) / "demo_deadlock.py"
    file_deadlock.write_text(skrip_deadlock, encoding="utf-8")

    try:
        hasil_deadlock = subprocess.run(
            [sys.executable, "-u", str(file_deadlock)],
            capture_output=True,
            text=True,
            timeout=3
        )
        print("Program selesai tanpa timeout.")
        print(hasil_deadlock.stdout)
    except subprocess.TimeoutExpired as e:
        keluaran = e.stdout
        if isinstance(keluaran, bytes):
            keluaran = keluaran.decode("utf-8", errors="replace")

        print("TIMEOUT setelah 3 detik - DEADLOCK TERKONFIRMASI!")
        print("Output sebelum timeout:")
        print(keluaran.strip() if keluaran else "Tidak ada output yang tertangkap.")

TIMEOUT setelah 3 detik - DEADLOCK TERKONFIRMASI!
Output sebelum timeout:
Thread1 memegang Lock-A
Thread2 memegang Lock-B
Thread3 memegang Lock-C
Thread3 menunggu Lock-A
Thread1 menunggu Lock-B
Thread2 menunggu Lock-C


Output timeout menunjukkan bahwa ketiga thread sudah memegang lock pertama, tetapi masing-masing menunggu lock yang sedang dipegang thread berikutnya. Tidak ada thread yang dapat melanjutkan dan melepaskan lock, sehingga terbentuk circular wait.

In [4]:
# Perbaikan deadlock dengan urutan pengambilan lock yang konsisten
lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()

def kerja_aman(nama):
    # Semua thread wajib mengambil lock dengan urutan A -> B -> C.
    with lock_a:
        print(f"[{nama}] memperoleh Lock-A")
        time.sleep(0.05)
        with lock_b:
            print(f"[{nama}] memperoleh Lock-B")
            with lock_c:
                print(f"[{nama}] memperoleh Lock-C dan menyelesaikan pekerjaan")

threads_aman = [
    threading.Thread(target=kerja_aman, args=(f"Thread-{i}",))
    for i in range(1, 4)
]

for thread in threads_aman:
    thread.start()
for thread in threads_aman:
    thread.join()

print("\nSELESAI - semua thread berhasil tanpa deadlock.")

[Thread-1] memperoleh Lock-A
[Thread-1] memperoleh Lock-B
[Thread-1] memperoleh Lock-C dan menyelesaikan pekerjaan
[Thread-2] memperoleh Lock-A
[Thread-2] memperoleh Lock-B
[Thread-2] memperoleh Lock-C dan menyelesaikan pekerjaan
[Thread-3] memperoleh Lock-A
[Thread-3] memperoleh Lock-B
[Thread-3] memperoleh Lock-C dan menyelesaikan pekerjaan

SELESAI - semua thread berhasil tanpa deadlock.


Penjelasan:

- Pola awal memenuhi empat syarat Coffman: *mutual exclusion*, *hold and wait*, *no preemption*, dan *circular wait*.
- Pada perbaikan, semua thread mengambil lock dengan urutan yang sama, yaitu Lock-A, Lock-B, lalu Lock-C. Thread yang belum mendapat Lock-A tidak memegang lock lain, sehingga rantai tunggu melingkar tidak dapat terbentuk.
- Timeout dipakai hanya pada proses demonstrasi deadlock agar proses notebook utama tetap dapat melanjutkan eksekusi.

3. **Bounded Buffer dengan Kapasitas Berbeda:** Ulangi eksperimen bagian 7 pada modul dengan `UKURAN_BUFFER = 1, 3, 10` dan dua producer serta dua consumer sekaligus.


In [5]:
# Simulasi Bounded Buffer: perbandingan kapasitas 1, 3, dan 10
JMLAH_PER_PRODUCER = 8
JEDA_PRODUCER = 0.01
JEDA_CONSUMER = 0.08
ringkasan_buffer = []

def eksperimen_buffer(ukuran_buffer):
    buffer = []
    mutex = threading.Lock()
    slot_kosong = threading.Semaphore(ukuran_buffer)
    slot_terisi = threading.Semaphore(0)

    total_item = 2 * JMLAH_PER_PRODUCER
    statistik = {
        "maks_terisi": 0,
        "diproduksi": 0,
        "dikonsumsi": 0,
        "producer_menunggu": 0
    }

    def producer(nama):
        for i in range(JMLAH_PER_PRODUCER):
            item = f"{nama}-item-{i + 1}"
            time.sleep(JEDA_PRODUCER)

            # Jika tidak ada slot kosong, producer harus menunggu.
            if not slot_kosong.acquire(blocking=False):
                with mutex:
                    statistik["producer_menunggu"] += 1
                slot_kosong.acquire()

            with mutex:
                buffer.append(item)
                statistik["diproduksi"] += 1
                statistik["maks_terisi"] = max(
                    statistik["maks_terisi"], len(buffer)
                )
                print(
                    f"[{nama}] taruh {item:<18} "
                    f"(buffer: {len(buffer)}/{ukuran_buffer})"
                )

            slot_terisi.release()

    def consumer(nama):
        for _ in range(total_item // 2):
            slot_terisi.acquire()

            with mutex:
                item = buffer.pop(0)
                statistik["dikonsumsi"] += 1
                print(
                    f"[{nama}] ambil {item:<18} "
                    f"(buffer: {len(buffer)}/{ukuran_buffer})"
                )

            slot_kosong.release()
            time.sleep(JEDA_CONSUMER)

    print(f"\n===== UKURAN_BUFFER = {ukuran_buffer} =====")

    threads = [
        threading.Thread(target=producer, args=("Producer-1",)),
        threading.Thread(target=producer, args=("Producer-2",)),
        threading.Thread(target=consumer, args=("Consumer-1",)),
        threading.Thread(target=consumer, args=("Consumer-2",))
    ]

    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

    print(
        f"Selesai: diproduksi={statistik['diproduksi']}, "
        f"dikonsumsi={statistik['dikonsumsi']}, sisa buffer={len(buffer)}"
    )
    print(
        f"Isi maksimum buffer: {statistik['maks_terisi']}/{ukuran_buffer} | "
        f"Producer tertahan: {statistik['producer_menunggu']} kali"
    )

    ringkasan_buffer.append({
        "Kapasitas Buffer": ukuran_buffer,
        "Total Diproduksi": statistik["diproduksi"],
        "Total Dikonsumsi": statistik["dikonsumsi"],
        "Isi Maksimum Buffer": statistik["maks_terisi"],
        "Producer Menunggu": statistik["producer_menunggu"],
        "Sisa Buffer": len(buffer)
    })


for ukuran in [1, 3, 10]:
    eksperimen_buffer(ukuran)

print("\nRingkasan eksperimen:")
print(
    f"{'Kapasitas':<12}{'Diproduksi':<13}{'Dikonsumsi':<13}"
    f"{'Isi Maks.':<12}{'Producer tunggu':<18}{'Sisa':<6}"
)
for data in ringkasan_buffer:
    print(
        f"{data['Kapasitas Buffer']:<12}"
        f"{data['Total Diproduksi']:<13}"
        f"{data['Total Dikonsumsi']:<13}"
        f"{data['Isi Maksimum Buffer']:<12}"
        f"{data['Producer Menunggu']:<18}"
        f"{data['Sisa Buffer']:<6}"
    )


===== UKURAN_BUFFER = 1 =====
[Producer-1] taruh Producer-1-item-1  (buffer: 1/1)
[Consumer-1] ambil Producer-1-item-1  (buffer: 0/1)
[Producer-2] taruh Producer-2-item-1  (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-1  (buffer: 0/1)
[Producer-1] taruh Producer-1-item-2  (buffer: 1/1)


[Consumer-1] ambil Producer-1-item-2  (buffer: 0/1)
[Producer-2] taruh Producer-2-item-2  (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-2  (buffer: 0/1)
[Producer-1] taruh Producer-1-item-3  (buffer: 1/1)
[Consumer-1] ambil Producer-1-item-3  (buffer: 0/1)
[Producer-2] taruh Producer-2-item-3  (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-3  (buffer: 0/1)
[Producer-1] taruh Producer-1-item-4  (buffer: 1/1)


[Consumer-1] ambil Producer-1-item-4  (buffer: 0/1)
[Producer-2] taruh Producer-2-item-4  (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-4  (buffer: 0/1)
[Producer-1] taruh Producer-1-item-5  (buffer: 1/1)


[Consumer-1] ambil Producer-1-item-5  (buffer: 0/1)
[Producer-2] taruh Producer-2-item-5  (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-5  (buffer: 0/1)
[Producer-1] taruh Producer-1-item-6  (buffer: 1/1)
[Consumer-1] ambil Producer-1-item-6  (buffer: 0/1)
[Producer-2] taruh Producer-2-item-6  (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-6  (buffer: 0/1)
[Producer-1] taruh Producer-1-item-7  (buffer: 1/1)


[Consumer-1] ambil Producer-1-item-7  (buffer: 0/1)
[Producer-2] taruh Producer-2-item-7  (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-7  (buffer: 0/1)
[Producer-1] taruh Producer-1-item-8  (buffer: 1/1)


[Consumer-1] ambil Producer-1-item-8  (buffer: 0/1)
[Producer-2] taruh Producer-2-item-8  (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-8  (buffer: 0/1)
Selesai: diproduksi=16, dikonsumsi=16, sisa buffer=0
Isi maksimum buffer: 1/1 | Producer tertahan: 13 kali

===== UKURAN_BUFFER = 3 =====
[Producer-1] taruh Producer-1-item-1  (buffer: 1/3)
[Consumer-1] ambil Producer-1-item-1  (buffer: 0/3)
[Producer-2] taruh Producer-2-item-1  (buffer: 1/3)
[Consumer-2] ambil Producer-2-item-1  (buffer: 0/3)
[Producer-2] taruh Producer-2-item-2  (buffer: 1/3)
[Producer-1] taruh Producer-1-item-2  (buffer: 2/3)
[Producer-2] taruh Producer-2-item-3  (buffer: 3/3)


... (7 outputs left out)

[Consumer-2] ambil Producer-2-item-4  (buffer: 9/10)
[Consumer-1] ambil Producer-1-item-4  (buffer: 8/10)
[Consumer-2] ambil Producer-1-item-5  (buffer: 7/10)
[Consumer-1] ambil Producer-2-item-5  (buffer: 6/10)


[Consumer-2] ambil Producer-2-item-6  (buffer: 5/10)
[Consumer-1] ambil Producer-1-item-6  (buffer: 4/10)


[Consumer-1] ambil Producer-1-item-7  (buffer: 3/10)
[Consumer-2] ambil Producer-2-item-7  (buffer: 2/10)
[Consumer-1] ambil Producer-1-item-8  (buffer: 1/10)
[Consumer-2] ambil Producer-2-item-8  (buffer: 0/10)


Selesai: diproduksi=16, dikonsumsi=16, sisa buffer=0
Isi maksimum buffer: 10/10 | Producer tertahan: 4 kali

Ringkasan eksperimen:
Kapasitas   Diproduksi   Dikonsumsi   Isi Maks.   Producer tunggu   Sisa  
1           16           16           1           13                0     
3           16           16           3           11                0     
10          16           16           10          4                 0     


Penjelasan:

1. **Buffer berkapasitas 1:** Buffer hanya dapat menyimpan satu item. Producer lebih sering tertahan ketika slot terisi, sehingga proses producer dan consumer cenderung bergantian.
2. **Buffer berkapasitas 3:** Beberapa item dapat menunggu di buffer sekaligus. Producer mempunyai ruang lebih besar sebelum harus menunggu, sehingga output dapat memperlihatkan beberapa operasi producer berurutan sebelum consumer mengambil item.
3. **Buffer berkapasitas 10:** Buffer dapat menampung lebih banyak item. Producer bisa mengisi beberapa slot lebih dahulu, sehingga consumer dapat mengambil item yang sudah mengantre.

Kapasitas buffer memengaruhi banyaknya item yang dapat menunggu dan frekuensi producer tertahan, bukan jumlah total item yang diproses. Jumlah producer yang tertahan dapat dilihat pada tabel ringkasan. Umumnya, buffer yang lebih besar mengurangi frekuensi producer harus menunggu slot kosong. Pada ketiga percobaan, total produksi dan konsumsi tetap sama serta buffer kembali kosong. Urutan output dan jumlah waktu tunggu dapat sedikit berbeda pada eksekusi berikutnya karena penjadwalan thread.

4. **Refleksi singkat (maks. 150 kata):** Dari deadlock, livelock, dan starvation, mana yang paling berbahaya untuk aplikasi produksi dan mana yang paling sulit dideteksi saat testing?

Menurutku, **deadlock paling berbahaya** untuk aplikasi produksi, terutama perbankan atau e-commerce. Ketika thread saling menunggu lock, transaksi terkait tidak dapat dilanjutkan. Jika terjadi pada proses penting, antrean transaksi dapat menumpuk dan layanan terganggu. Livelock juga membuang sumber daya karena thread tetap aktif tanpa menghasilkan kemajuan, sedangkan starvation membuat sebagian request terus kalah mendapatkan giliran.

Menurutku, **starvation paling sulit dideteksi saat testing**. Aplikasi masih dapat berjalan dan sebagian request tetap berhasil, sehingga masalahnya tidak selalu terlihat sebagai sistem yang berhenti total. Pengujian singkat atau beban rendah mungkin tidak menunjukkan request yang terus tertunda. Karena itu, pengujian perlu mencakup beban tinggi, pengukuran waktu tunggu, dan pemantauan keadilan akses resource.